# 00 — Setup: Corpus, Requirements, Dockerfile

Run this notebook **first**. It materializes everything that isn't pipeline
Python logic but still needs to exist as a real file on disk:

- `docs/doc_01.txt` … `doc_08.txt` — the exact 8 Zepto policy documents given in
  the assignment spec (kept as `.txt`, as the spec explicitly requires: *"copy it
  into your repository as-is"*).
- `requirements.txt` — pinned dependency list.
- `Dockerfile` — kept literally named `Dockerfile` because `docker build` looks
  for that exact filename; it cannot be a `.ipynb`.

Everything else in this project (`ingest.py`, `prompt_template.py`, `schemas.py`,
`graph.py`, `main.py`) is generated by the other numbered notebooks
(`01_ingest.ipynb` … `04_app.ipynb`), keeping all *logic* authored in `.ipynb`
form as required.


In [ ]:
import os
os.makedirs("docs", exist_ok=True)
print("docs/ ready")
 

In [ ]:
%%writefile docs/doc_01.txt
Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.


In [ ]:
%%writefile docs/doc_02.txt
Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3-5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.


In [ ]:
%%writefile docs/doc_03.txt
Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.


In [ ]:
%%writefile docs/doc_04.txt
Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.


In [ ]:
%%writefile docs/doc_05.txt
Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.


In [ ]:
%%writefile docs/doc_06.txt
If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed.


In [ ]:
%%writefile docs/doc_07.txt
Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.


In [ ]:
%%writefile docs/doc_08.txt
Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered.


In [ ]:
%%writefile requirements.txt
# Core pipeline
langgraph>=0.2.0
chromadb>=0.5.0
sentence-transformers>=3.0.0
fastapi>=0.110.0
uvicorn[standard]>=0.29.0
pydantic>=2.6.0
requests>=2.31.0

# Notebook execution (source lives in .ipynb; Docker build materializes the .py
# modules from them via `jupyter nbconvert --execute`)
jupyter>=1.0.0
nbconvert>=7.16.0
ipykernel>=6.29.0

# Optional MOCK_LLM=0 extension only (Groq's API is OpenAI-compatible)
openai>=1.30.0


In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

# System deps for sentence-transformers / chromadb wheels
RUN apt-get update && apt-get install -y --no-install-recommends build-essential \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY docs/ ./docs/
COPY 01_ingest.ipynb 02_prompt_template.ipynb 03_graph.ipynb 04_app.ipynb ./

# Source of truth is the notebooks (per assignment requirement). Executing them
# here (in dependency order) runs their %%writefile cells and materializes the
# .py modules (ingest.py, prompt_template.py, schemas.py, graph.py, main.py)
# that uvicorn needs, and builds the ChromaDB collection under ./chroma_db.
# MOCK_LLM defaults to mock mode, so this build step makes no LLM/network calls
# beyond the one-time embedding-model download.
RUN jupyter nbconvert --to notebook --execute --inplace 01_ingest.ipynb && \
    jupyter nbconvert --to notebook --execute --inplace 02_prompt_template.ipynb && \
    jupyter nbconvert --to notebook --execute --inplace 03_graph.ipynb && \
    jupyter nbconvert --to notebook --execute --inplace 04_app.ipynb

EXPOSE 7860

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]


In [ ]:
import os
for f in sorted(os.listdir("docs")):
    print("docs/" + f)
print("requirements.txt" if os.path.exists("requirements.txt") else "MISSING requirements.txt")
print("Dockerfile" if os.path.exists("Dockerfile") else "MISSING Dockerfile")
